# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [ ]:
import pandas as pd  # brings in pandas for table-style data work
import numpy as np  # gives us NumPy tools for numeric checks

# Option A: Upload in Colab
# from google.colab import files  # use this import if you are running in Colab
# uploaded = files.upload()  # opens a file picker so you can upload the CSV by hand
# df = pd.read_csv('SampleSuperstore.csv')  # reads the uploaded file into a DataFrame

# Option B: Load from a public URL
# If the URL does not work, use Option A.
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"  # stores the dataset link in one place

try:  # tries the online file first
    df = pd.read_csv(url, encoding='latin-1')  # loads the CSV from the link
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")  # confirms the dataset size
except Exception as e:  # catches any loading error and switches to manual upload
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")  # explains what went wrong
    # Fallback: upload manually
    from google.colab import files  # imports the upload helper only if it is needed
    uploaded = files.upload()  # lets you choose the file from your computer
    filename = list(uploaded.keys())[0]  # grabs the uploaded file name
    df = pd.read_csv(filename, encoding='latin-1')  # reads the uploaded CSV
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")  # confirms the fallback load worked

---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [ ]:
# First 5 rows
df.head()  # shows the first five rows for a quick glance

In [ ]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")  # prints how big the dataset is

In [ ]:
# Data types and non-null counts
df.info()  # lists columns, types, and filled-in values

In [ ]:
# Summary statistics for numerical columns
df.describe()  # summarizes the numeric columns with common stats

---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [ ]:
# Count missing values per column
missing = df.isnull().sum()  # counts null values in each column
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)  # turns those counts into percentages

missing_report = pd.DataFrame({  # builds a small report table
    'missing_count': missing,  # keeps the raw missing-value counts
    'missing_pct': missing_pct  # keeps the percentage for each column
})

# Show only columns with missing values
missing_report[missing_report['missing_count'] > 0]  # filters the report down to columns that need attention

In [ ]:
# Fill missing numerical values with median (robust to outliers)
numerical_cols = df.select_dtypes(include=[np.number]).columns  # collects every numeric column name
for col in numerical_cols:  # checks each numeric column one by one
    if df[col].isnull().sum() > 0:  # only works on columns that actually have gaps
        median_val = df[col].median()  # finds the middle value for that column
        df[col].fillna(median_val, inplace=True)  # fills nulls with the median
        print(f"Filled {col} missing values with median: {median_val}")  # reports what was used

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns  # collects the text-style columns
for col in categorical_cols:  # checks each categorical column
    if df[col].isnull().sum() > 0:  # skips columns that are already complete
        mode_val = df[col].mode()[0]  # picks the most common value
        df[col].fillna(mode_val, inplace=True)  # fills nulls with that common value
        print(f"Filled {col} missing values with mode: {mode_val}")  # reports the fill choice

# Verify: no more missing values
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")  # checks that the cleanup is finished

---
## Task 3: Remove Duplicates (pre-built)

In [ ]:
# Check for duplicates
n_duplicates = df.duplicated().sum()  # counts exact duplicate rows
print(f"Duplicate rows found: {n_duplicates}")  # shows how many duplicates were found

# Remove duplicates
if n_duplicates > 0:  # only removes rows if duplicates exist
    df = df.drop_duplicates()  # keeps the first copy and drops the rest
    print(f"After removal: {df.shape[0]} rows remain")  # shows the new row count
else:  # runs when the dataset is already clean
    print("No duplicates to remove.")  # makes it clear that nothing changed

---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [ ]:
# Check current types of date columns
print("Before conversion:")  # adds a heading to the output
for col in df.columns:  # looks through every column name
    if 'date' in col.lower() or 'Date' in col:  # keeps only the date-related columns
        print(f"  {col}: {df[col].dtype}")  # shows the current data type
        print(f"  Sample value: {df[col].iloc[0]}")  # shows one example value from that column

In [ ]:
date_cols = [col for col in df.columns if 'date' in col.lower()]  # gathers every date-like column name
for col in date_cols:  # converts each date column in turn
    df[col] = pd.to_datetime(df[col], errors='coerce')  # turns text into datetime values
    print(f"Converted {col} to datetime")  # confirms the conversion

In [ ]:
print("After conversion:")  # adds a heading for the check
for col in date_cols:  # loops through the converted date columns
    print(f"  {col}: {df[col].dtype}")  # shows the new data type for each one

---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [ ]:
# First, identify the right column names
print("All columns:")  # prints a heading before the list
for col in df.columns:  # walks through every column name
    print(f"  {col}")  # prints one column name per line

In [ ]:
normalize = lambda name: name.lower().replace(' ', '').replace('_', '')  # smooths out small naming differences
customer_col = next(col for col in df.columns if normalize(col) == 'customerid')  # finds the customer ID column
sales_col = next(col for col in df.columns if normalize(col) == 'sales')  # finds the sales column
customer_spending = df.groupby(customer_col)[sales_col].sum()  # totals sales for each customer
customer_spending.name = 'total_spending'  # gives the result a clear name

In [ ]:
order_freq = df.groupby(customer_col).size()  # counts how many orders each customer placed
order_freq.name = 'order_frequency'  # labels the result clearly

In [ ]:
avg_order = df.groupby(customer_col)[sales_col].mean()  # calculates the average order value per customer
avg_order.name = 'avg_order_value'  # labels the result clearly

In [ ]:
customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1).reset_index()  # combines the customer metrics into one table

In [ ]:
print("First 10 rows:")  # labels the preview output
display(customer_summary.head(10))  # shows the first ten customer records
print("\nSummary statistics:")  # labels the descriptive stats
display(customer_summary.describe())  # shows summary stats for the customer features

---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [ ]:
df.to_csv('superstore_cleaned.csv', index=False)  # saves the cleaned order-level data
customer_summary.to_csv('customer_summary.csv', index=False)  # saves the customer-level summary
print("Saved superstore_cleaned.csv and customer_summary.csv")  # confirms both files were written

---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?